[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_15/3_stats.ipynb)

# Day 15: Statistics, A/B testing and product (medium)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (26 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | The standard error of average order value | medium | 7 |
| Q2 | Prove it: 1,000 fake experiments | medium | 7 |
| Q3 | Which average order value? | medium | 6 |
| Q4 | The UK looks different (review) | medium | 6 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `retail` (UCI Online Retail, real, 541,909 invoice lines from a UK web shop, 2010-12 to 2011-12): `InvoiceNo` (a C prefix means a cancellation), `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID` (about 25% missing), `Country`.

- `ab` (Udacity e-commerce landing page test, real, 294,478 rows): `user_id`, `timestamp`, `group` (control / treatment), `landing_page` (old_page / new_page), `converted` (0/1). It has some rows where group and page do not match and some repeated users.

- `countries` (290,584 rows): `user_id`, `country` (UK, US, CA). Joins to `ab`.

In Colab, if `online_retail.csv.gz` is not in your Drive data folder, upload it there (it is converted from the UCI xlsx, so there is no direct CSV link). The setup builds two tables:

- `orders`: one row per order: `CustomerID`, `InvoiceNo`, `rev` (order revenue, capped at the 99th percentile).
- `users`: one row per customer: `rev` (total revenue), `n_orders`.

Think of every customer as a user in an experiment: randomisation is by customer, the metric is average order value (AOV) = total revenue / total orders.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
retail = pd.read_csv(data_path("online_retail.csv.gz"), dtype={"InvoiceNo": str, "StockCode": str},
                     parse_dates=["InvoiceDate"])
ab = pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv"))
countries = pd.read_csv(data_path("ab_countries.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/countries.csv"))
print("loaded:", "retail", retail.shape, "ab", ab.shape, "countries", countries.shape)
# Today's tables (built from the real retail data):
# valid purchases only (no cancellations, known customer), one row per order, order revenue capped at the 99th
# percentile so a few giant wholesale orders do not dominate every number.
ok = retail.CustomerID.notna() & ~retail.InvoiceNo.str.startswith("C") & (retail.Quantity > 0) & (retail.UnitPrice > 0)
orders = (retail[ok].assign(rev=lambda d: d.Quantity * d.UnitPrice)
          .groupby(["CustomerID", "InvoiceNo"], as_index=False).rev.sum())
orders["rev"] = orders.rev.clip(upper=orders.rev.quantile(0.99))
users = orders.groupby("CustomerID").agg(rev=("rev", "sum"), n_orders=("rev", "size"))
print("orders", orders.shape, "users", users.shape)

---
### Q1. The standard error of average order value

*medium, about 7 min*

AOV is `sum(revenue) / sum(orders)` over all customers. Randomisation would be by customer, so customers (not orders) are the independent units.

(a) Compute AOV.

(b) 'Naive' SE: treat the 18,532 orders as independent, `sd(order revenue) / sqrt(#orders)`.

(c) Delta-method SE with customers as units. With `Y` = revenue per customer, `N` = orders per customer, `R = mean(Y) / mean(N)`, n customers: `Var(R) = (var(Y) - 2 R cov(Y, N) + R^2 var(N)) / (n * mean(N)^2)`.

(d) How many times larger is the delta SE? Say it to a PM.

Follow-ups: when would the two SEs be about equal? Name a ratio metric where the unit of analysis is clearly not the unit of randomisation in a video app.

In [ ]:
Y, N = users.rev.values, users.n_orders.values
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a metric that is a ratio of two sums, analysed at a finer level (orders) than the randomisation unit (customers). Method: the delta method (or a bootstrap over customers).

</details>

<details><summary><b>Hint 2</b></summary>

1. `R = Y.mean() / N.mean()`. 2. Naive: `orders.rev.std() / sqrt(len(orders))`. 3. Delta: use `np.var(.., ddof=1)` and `np.cov(Y, N)[0, 1]` in the formula. 4. Ratio of the two SEs.

</details>

<details><summary><b>Solution</b></summary>

```python
Y, N = users.rev.values, users.n_orders.values

def delta_ratio(Y, N):
    """Ratio of means and its delta-method variance (units = rows of Y and N)."""
    n, mY, mN = len(Y), Y.mean(), N.mean()
    R = mY / mN
    var = (Y.var(ddof=1) - 2 * R * np.cov(Y, N)[0, 1] + R ** 2 * N.var(ddof=1)) / (n * mN ** 2)
    return R, var

R, var = delta_ratio(Y, N)
naive = orders.rev.std() / np.sqrt(len(orders))
print(f"AOV {R:.1f}, naive SE {naive:.2f}, delta SE {np.sqrt(var):.2f}, ratio {np.sqrt(var) / naive:.1f}")
# AOV 428.8, naive SE 3.95, delta SE 14.70, ratio 3.7
```

**Why:** Orders from the same customer are alike (a wholesaler always places big orders), so 18,532 orders carry much less independent information than their count suggests. The delta method linearises the ratio around the means and uses customer-level variances and the covariance of revenue and order count. Here the honest SE is 3.7 times the naive one, so a naive test would have confidence intervals 3.7 times too narrow.

**Say it to a PM:** Average order value is about 429; because big customers place many similar orders, our uncertainty is almost 4 times larger than a simple order-level calculation suggests, so we must use the customer-level method or we will declare false wins.

**Follow-ups:**

- *When are they equal?* When orders within a customer are no more alike than orders across customers (no clustering), for example if every customer placed exactly one order.
- *Video app example?* Click-through rate = clicks / impressions with users randomised; average watch time per video view; likes per session. All are ratios over a unit (impression, view, session) finer than the user.

**Complexity:** O(number of customers) once the per-customer sums exist.

**Common mistakes:** Running a t-test on order-level rows. Forgetting the covariance term (Y and N are strongly correlated, and dropping it changes the answer a lot). Using population variance in one place and sample variance in another (minor).

**Learn more:** [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics)

</details>

---
### Q2. Prove it: 1,000 fake experiments

*medium, about 7 min*

Run 1,000 A/A tests: each time, assign every customer to A or B at random (50/50, seed 0) and test the difference in AOV with (i) the naive order-level z-test and (ii) the delta-method z-test `z = (R_A - R_B) / sqrt(Var(R_A) + Var(R_B))`.

(a) What share of A/A tests is 'significant' at 0.05 for each method?

(b) Say it to a PM.

Follow-up: what other method gives a correct SE here without any formula?

In [ ]:
rng = np.random.default_rng(0)
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: check a variance formula by simulation. Method: A/A tests with random splits of the real units; a correct test is 'significant' about 5% of the time.

</details>

<details><summary><b>Hint 2</b></summary>

1. Map each order to its customer's position. 2. Each run: `g = rng.integers(0, 2, n_users).astype(bool)`. 3. Delta: `delta_ratio` on each half. 4. Naive: order revenues of each half, Welch-style z. 5. Count |z| > 1.96.

</details>

<details><summary><b>Solution</b></summary>

```python
def delta_ratio(Y, N):
    n, mY, mN = len(Y), Y.mean(), N.mean()
    R = mY / mN
    return R, (Y.var(ddof=1) - 2 * R * np.cov(Y, N)[0, 1] + R ** 2 * N.var(ddof=1)) / (n * mN ** 2)

rng = np.random.default_rng(0)
Y, N = users.rev.values, users.n_orders.values
pos = orders.CustomerID.map(pd.Series(np.arange(len(users)), index=users.index)).values
o_rev = orders.rev.values
hits_naive = hits_delta = 0
sims = 1000
for _ in range(sims):
    g = rng.integers(0, 2, len(users)).astype(bool)
    Ra, va = delta_ratio(Y[g], N[g])
    Rb, vb = delta_ratio(Y[~g], N[~g])
    hits_delta += abs(Ra - Rb) / np.sqrt(va + vb) > 1.96
    a, b = o_rev[g[pos]], o_rev[~g[pos]]
    z = (a.mean() - b.mean()) / np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    hits_naive += abs(z) > 1.96
print("false positive rate naive:", hits_naive / sims, " delta:", hits_delta / sims)
# false positive rate naive: 0.611  delta: 0.051
```

**Why:** A correct 5% test should flag about 5% of A/A splits. The naive order-level test flags 61% of them, because its SE is about 3.7 times too small, so `|z|` is inflated by the same factor. The delta method lands at 5.1%.

**Say it to a PM:** When we pretend orders are independent, more than half of experiments with no real change would look like wins; with the customer-level method about 1 in 20 does, which is what we promise.

**Follow-ups:**

- *Without a formula?* A bootstrap that resamples *customers* (with all their orders), recomputing AOV each time. Or compute per-customer AOV contributions and use a cluster-robust regression of order revenue on treatment with customer clusters, which gives the same answer as the delta method.

**Complexity:** O(sims * n_orders), a few seconds for 1,000 runs.

**Common mistakes:** Resampling orders instead of customers in a bootstrap (same mistake as the naive test). Too few A/A runs to see the rate (with 1,000 runs the 5% estimate has a standard error of about 0.7 pp).

**Learn more:** [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
### Q3. Which average order value?

*medium, about 6 min*

(a) Compute two versions of 'average order value': the ratio of sums `sum(rev) / sum(orders)` and the mean of per-customer ratios `mean(rev_i / orders_i)`. Why do they differ?

(b) A treatment adds free shipping on small orders. Toy numbers (made up): in control each user places 2 orders of 50; in treatment each user places the same 2 orders plus one extra order of 20. Compute AOV and revenue per user in each arm.

(c) Say it to a PM: which metric should decide the launch?

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: an average of ratios vs a ratio of totals, and a denominator the treatment can change. Concept: ratio metric definition and denominator shift.

</details>

<details><summary><b>Hint 2</b></summary>

1. `users.rev.sum() / users.n_orders.sum()` vs `(users.rev / users.n_orders).mean()`. 2. Toy: AOV = revenue / orders per arm; revenue per user = revenue / users.

</details>

<details><summary><b>Solution</b></summary>

```python
ratio_of_sums = users.rev.sum() / users.n_orders.sum()
mean_of_ratios = (users.rev / users.n_orders).mean()
print(f"ratio of sums {ratio_of_sums:.1f}, mean of ratios {mean_of_ratios:.1f}")
# ratio of sums 428.8, mean of ratios 371.2

# toy: 1 user per arm is enough to see it
aov_c, rpu_c = (50 + 50) / 2, 50 + 50
aov_t, rpu_t = (50 + 50 + 20) / 3, 50 + 50 + 20
print(f"control AOV {aov_c:.0f}, revenue/user {rpu_c}; treatment AOV {aov_t:.0f}, revenue/user {rpu_t}")
# control AOV 50, revenue/user 100; treatment AOV 40, revenue/user 120
```

**Why:** The ratio of sums weights each customer by their number of orders, so frequent buyers (who here place larger orders) count more; the mean of ratios gives every customer one vote. Neither is wrong, but they answer different questions: 'the typical order' vs 'the typical customer's basket'. In the toy case AOV drops 20% while revenue per user rises 20%: the treatment changed the denominator (more, smaller orders), so AOV alone would kill a good feature.

**Say it to a PM:** Average order value went down because people added small extra orders, but each user spends 20% more, so we should judge the launch on revenue per user and keep AOV as a diagnostic.

**Follow-ups:**

- *Rule of thumb* Pick as the decision metric a per-randomisation-unit metric (revenue per user). Use ratio metrics such as AOV or CTR to explain *why*, and always check whether the treatment moved the denominator (orders, impressions) first.

**Complexity:** O(n).

**Common mistakes:** Mixing the two definitions between dashboards. Declaring a loss because AOV fell when the number of orders rose.

**Learn more:** [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics), [stats-product-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-product-metrics)

</details>

---
### Q4. The UK looks different (review)

*medium, about 6 min*

Join the cleaned `ab` table (drop group/page mismatches, keep the first row per user) with `countries`. A stakeholder wants the landing page result by country.

(a) For each country: conversion in each arm, difference, and the two-proportion z-test p-value.

(b) Is any country significant after a Bonferroni correction for 3 tests? Note which country has a positive difference.

(c) Say it to a PM who wants to launch the new page in the UK only.

In [ ]:
ok = (ab.group == "treatment") == (ab.landing_page == "new_page")
clean = ab[ok].sort_values("timestamp").drop_duplicates("user_id").merge(countries, on="user_id")
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: one test sliced into segments after the fact. Pitfall: multiple testing and post-hoc segment picking. Correct alpha (Bonferroni 0.05/3) and look at the CIs.

</details>

<details><summary><b>Hint 2</b></summary>

1. `groupby(['country', 'group']).converted.agg(['sum', 'count'])`. 2. z-test per country. 3. Compare each p with 0.05/3 = 0.0167.

</details>

<details><summary><b>Solution</b></summary>

```python
ok = (ab.group == "treatment") == (ab.landing_page == "new_page")
clean = ab[ok].sort_values("timestamp").drop_duplicates("user_id").merge(countries, on="user_id")
for country, d in clean.groupby("country"):
    g = d.groupby("group").converted.agg(["sum", "count"])
    x_c, n_c = g.loc["control"]
    x_t, n_t = g.loc["treatment"]
    pool = (x_c + x_t) / (n_c + n_t)
    diff = x_t / n_t - x_c / n_c
    z = diff / np.sqrt(pool * (1 - pool) * (1 / n_c + 1 / n_t))
    print(f"{country}: control {x_c / n_c:.4f}, treatment {x_t / n_t:.4f}, diff {diff * 100:+.2f} pp, "
          f"p {2 * stats.norm.sf(abs(z)):.3f}")
# CA: control 0.1188, treatment 0.1119, diff -0.69 pp, p 0.195
# UK: control 0.1200, treatment 0.1212, diff +0.11 pp, p 0.635
# US: control 0.1206, treatment 0.1185, diff -0.22 pp, p 0.132
```

**Why:** No country is significant even before correction (all p > 0.13), and none comes close to 0.0167. The UK is the only country with a positive difference (+0.11 pp, p = 0.64), which is what you expect by chance when slicing 3 ways. Heterogeneity should be tested directly (an interaction term), not by eyeballing separate p-values.

**Say it to a PM:** No country shows a real effect; the UK's small plus is well within noise, so a UK-only launch would be based on luck. If we believe the UK is different, we should run a new test there with that hypothesis written down first.

**Follow-ups:**

- *How to test 'UK is different'?* Logistic regression `converted ~ treatment * country` and test the interaction terms jointly, or a chi-square test of homogeneity of the effects. Plan segments before the test.

**Complexity:** O(n log n) for the sort and merge.

**Common mistakes:** Comparing p-values between segments ('UK p = 0.64 vs US p = 0.13, so the effect differs'): a difference in significance is not a significant difference.

**Learn more:** [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls), [stats-heterogeneous-effects](https://mahsa-agz.github.io/ds-handbook/#stats-heterogeneous-effects)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_15/4_ai.ipynb)